In [1]:

import json
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

%load_ext autoreload
%autoreload 2


load_dotenv()
root_path = Path().absolute().parent.parent.parent
sys.path.append(str(root_path))


from functions.llm_config import LLMConfig
from functions.retrieval import QuestionRetriever
from functions.query_decomposer import QueryDecomposer
from functions.sqldatabase_langchain_utils import SQLDatabaseLangchainUtils
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
import kaggle_paths as paths

import re
from pydantic import BaseModel, Field
from typing import List



from eval_agent.text2sql_agent.text_to_sql.kaggle_text_to_sql_extended_schema import TextToSQLExtendedSchema




## KAGGLE

In [2]:

O3 = LLMConfig(provider="azure").get_llm(model="o3-mini", reasoning_effort="high")
GPT4O = LLMConfig(provider="azure").get_llm(model="gpt-4o")



decomposer = QueryDecomposer(
    GPT4O,
    paths.PROMPT_DECOMPOSER_FILE,
    False
)



experiment = "kaggle"

if "kaggle" in experiment:
    prompt_path = paths.KAGGLE_EXTENDED_SCHEMA_PROMPT
    # DATASET_SYNTHETIC_PATH = paths.DATASET_SYNTHETIC_PATH
    # EMBEDDINGS_PATH = paths.EMBEDDINGS_PATH

else:
    raise ValueError("Invalid experiment name. Use 'kaggle' .")


with open(paths.DB_CONNECTION_FILE, "r") as f:
    db_connection = json.load(f)

# retriever = QuestionRetriever(
#     dataset_path=DATASET_SYNTHETIC_PATH,
#     vectors_path=EMBEDDINGS_PATH,
#     # vectorize=True
# )
# retriever.remove_duplicates()

/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/functions/query_decomposer.py:37: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  self.query_analyzer = LLMChain(llm=llm, prompt=self.prompt_builder, verbose=False)


In [4]:
sqldatabase = SQLDatabaseLangchainUtils(db_connection)

In [5]:
t =  TextToSQLExtendedSchema(GPT4O, decomposer, prompt_path, debug=False)


def test(question):
    print("Running Test")

    result = t.translate_text_to_sql(question)

    
    result.sql_query = result.sql_query.replace("```sql", "").replace("```", "")
    if not result.sql_query.strip().upper().startswith("SELECT"):
        result.sql_query = "SELECT " + result.sql_query
    if "DISTINCT" in result.sql_query:
        result.sql_query = result.sql_query.replace("DISTINCT", "")
    print("Result")
    sql = result.sql_query
    print(result.sql_query)
    print(result.schema_linking_tables)
    print(sqldatabase.run_in_database(sql))
    

In [7]:
test("british nuclear reactor")

Running Test
Result
SELECT name, country, status, reactortype, reactormodel, capacity, constructionstartat, operationalfrom, operationalto FROM KAGGLE.GEONUCLEARDATA_NUCLEAR_POWER_PLANTS WHERE LOWER(country) LIKE '%united kingdom%' OR LOWER(country) LIKE '%britain%' OR LOWER(country) LIKE '%england%'
['KAGGLE.GEONUCLEARDATA_NUCLEAR_POWER_PLANTS']
[('Oldbury-A1', 'United Kingdom of Great Britain and Northern Ireland', 'Shutdown', 'GCR', 'MAGNOX', 300, datetime.datetime(1962, 5, 1, 0, 0), datetime.datetime(1967, 12, 31, 0, 0), datetime.datetime(2012, 2, 29, 0, 0)), ('Oldbury-A2', 'United Kingdom of Great Britain and Northern Ireland', 'Shutdown', 'GCR', 'MAGNOX', 300, datetime.datetime(1962, 5, 1, 0, 0), datetime.datetime(1968, 9, 30, 0, 0), datetime.datetime(2011, 6, 30, 0, 0)), ('Sizewell-A1', 'United Kingdom of Great Britain and Northern Ireland', 'Shutdown', 'GCR', 'MAGNOX', 290, datetime.datetime(1961, 4, 1, 0, 0), datetime.datetime(1966, 3, 25, 0, 0), datetime.datetime(2006, 12, 31

In [9]:
from eval_agent.text2sql_agent.tool_kaggle import convert_text_to_sql_and_execute

print(convert_text_to_sql_and_execute("Quais são os reatores nucleares no Reino Unido?"))

{'input': 'Quais são os reatores nucleares no Reino Unido?', 'schema_linking': ['KAGGLE.GEONUCLEARDATA_NUCLEAR_POWER_PLANTS'], 'answer': Empty DataFrame
Columns: [NAME, STATUS, REACTORTYPE, REACTORMODEL, CAPACITY]
Index: [], 'sql': "SELECT name, status, reactortype, reactormodel, capacity FROM KAGGLE.GEONUCLEARDATA_NUCLEAR_POWER_PLANTS WHERE LOWER(country) = LOWER('United Kingdom')"}


/home/tomas/PUC/TecGraf/conversational_bm/benchmark_nl_databases_conversational_interfaces/functions/dataset_utils.py:341: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(SQL_query, con=self.db_connection)
